# Notebook 14 — Cross-Species Generalisation

> **Supplementary §15**

Can the morphing shape modes learned from Harris' hawks serve as a shared coordinate system for other bird species? This notebook tests this idea by projecting hawk morphing modes onto cadaver-based morphological reconstructions of 21 avian species (Harvey et al. 2022b).

No flight kinematics data are available for these species — this is a **proof of concept** showing that the morphing shape modes provide a shared framework for comparing flight morphing across species, accommodating morphological differences.

## Approach

1. **Load and process** cadaver wing and body measurements (§15.1)
2. **Transform** the hawk mean shape to each species using piecewise, marker-specific linear transformations (§15.2)
3. **Transform the principal components** to the new morphology and animate the result (§15.3)

## Contents
1. [Setup and data loading](#setup)
2. [Load and process Harvey data](#harvey-data)
3. [Adjusting cadaver measurements](#adjustments)
4. [Transform hawk to target species](#transformation)
5. [Transform morphing modes and animate](#animate)
6. [Summary](#summary)

In [1]:
# --- Setup ---
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import pinv

plt.rcParams['font.family'] = 'Andale Mono'
np.set_printoptions(suppress=True, precision=3)

from morphing_birds import Animal3D, plot_plotly, animate_plotly

from kinematic_morphospace import (
    filter_by, run_PCA, reconstruct, to_bilateral,
    get_score_df, get_score_range, get_binned_scores,
    load_harvey_data, select_max_wingspan_row, clean_body_data,
    process_body_bird_id, merge_bird_data, filter_marker_columns,
    set_new_origin_and_axes, compute_derived_markers,
    check_and_fix_shoulder_distance, fix_leftright_sign,
    integrate_dataframe_to_bird3D,
    transform_hawk_to_species, transform_principal_components,
)
from kinematic_morphospace.plotting import (
    plot_bird_markers, plot_explained,
)

## Load Hawk PCA Data

We load the unilateral PCA results and the mean hawk shape. The hawk mean shape serves as the source morphology for the cross-species transformation.

In [2]:
unilateral_data = np.load("../../data/processed/unilateral_data.npy")
unilateral_frame_info_df = pd.read_csv("../../data/processed/unilateral_frame_info_df.csv")

principal_components = np.load("../../data/processed/unilateral_principal_components.npy")
scores = np.load("../../data/processed/unilateral_scores.npy")
mu = np.load("../../data/processed/unilateral_mu.npy").reshape(1, -1, 3)

# Reload PCA for variance ratios
filt = filter_by(unilateral_frame_info_df, obstacle=0)
_, _, pca = run_PCA(unilateral_data[filt], unilateral_data)

hawk3d = Animal3D('hawk', '../../data/mean_hawk_shape.csv')

print(f"Scores: {scores.shape}")
print(f"Components: {principal_components.shape}")
print(f"Mean shape: {mu.shape}")

Scores: (289528, 12)
Components: (12, 12)
Mean shape: (1, 4, 3)


## Load and Process Harvey Data

> **§15.1 — Adjusting Cadaver Measurements**

Cadaver measurements from Harvey et al. (2022b) provide wing and body landmark positions for 21 species. We select the maximum-wingspan posture for each specimen and derive marker positions corresponding to the hawk marker locations.

### Marker correspondence

The Harvey dataset uses numbered landmarks (`pt#`). Each is mapped to the closest anatomical equivalent of our hawk markers:

| Harvey landmark | Anatomical position | Hawk marker equivalent |
|---|---|---|
| pt9 | Wingtip (distal primary) | Wingtip |
| pt8 + pt4 (average) | Mid-hand wing | Primary |
| pt10 | Wrist / carpal joint | Secondary |
| pt11 | Tail root (base of tail) | Tail base |
| pt11 − tail length | Tail tip (derived) | Tailtip |
| pt2 / pt11 (pair) | Left / right shoulder | Shoulder (origin) |
| pt1 | Head (nape) | Hood |

### Key cadaver adjustments

Several adjustments are necessary because cadaver postures differ from live flight:

1. **Tail width doubled** — cadaver tails are furled; in flight, hawks spread their tails widely, particularly during slow manoeuvring. Doubling the cadaver tail width approximates its functional position.
2. **Shoulder distance** — adjusted to match the body-width measurement from the body dataset, ensuring the torso width is anatomically correct.
3. **Origin at mid-shoulders** — a central reference point equivalent to the backpack marker is computed as the average of the two shoulder landmarks (pt2 and pt11).

Steller's jay was excluded due to unreliable specimen measurements.

In [3]:
# Load Harvey wing and body data
wing_df, body_df = load_harvey_data(
    "../../data/2021_06_03_HARVEYallspecimen_winginfo.csv",
    "../../data/2021_06_03_HARVEYallspecimen_bodyinfo.csv",
)

print(f"Wing data: {wing_df.shape}")
print(f"Body data: {body_df.shape}")

Wing data: (70585, 47)
Body data: (938, 43)


In [4]:
# Select max wingspan per bird
wing_df = select_max_wingspan_row(
    wing_df, bird_id_col='BirdID', left_marker='pt8', right_marker='pt12'
)

# Clean body data
body_df = clean_body_data(body_df)

# Drop Steller's jay (specimen measurements are unreliable)
body_df = body_df[body_df['species_common'] != "Stellers_jay"]

# Process bird IDs
body_df = process_body_bird_id(body_df, id_col='bird_id')

# Filter to relevant markers
marker_names = ["pt1", "pt2", "pt4", "pt8", "pt9", "pt10", "pt11"]
base_columns = ["BirdID", "species"]
wing_df = filter_marker_columns(wing_df, marker_names, base_columns)

# Merge wing and body data
merged_df = merge_bird_data(wing_df, body_df, on_col='BirdID')

print(f"Merged data: {merged_df.shape}")
print(f"Species: {merged_df['species_common'].nunique()}")
print(merged_df[['species_common', 'wing_span_cm']].to_string(index=False))

Merged data: (35, 41)
Species: 21
        species_common  wing_span_cm
                merlin          54.1
                merlin          83.2
         western_grebe          78.6
      common_nighthawk          52.0
              barn_owl         106.0
              barn_owl         101.2
                pigeon          63.4
      great_blue_heron         184.4
               mallard          88.8
                pigeon          61.3
                pigeon          63.0
      great_blue_heron         182.5
  glaucous_winged_gull         132.4
      northern_flicker          51.3
          common_raven         116.0
         western_grebe          82.2
               mallard          90.5
                   NaN           NaN
    sharp_shinned_hawk          61.0
           black_swift          38.5
          storm_petrel          45.2
                merlin          62.5
          coopers_hawk          81.2
      common_nighthawk          55.0
      peregrine_falcon          83.9
    

In [5]:
# Set new coordinate system (origin at mid-shoulders)
merged_df = set_new_origin_and_axes(
    merged_df, origin_marker=['pt2', 'pt11'],
    origin_axes=('x',), new_axes=('y', 'x', '-z')
)

# Compute derived markers (wingtip, primary, secondary, etc.)
merged_df = compute_derived_markers(merged_df)

# Fix shoulder distance to match body measurements
merged_df = check_and_fix_shoulder_distance(merged_df)

# Correct left/right sign convention after shoulder adjustments
merged_df = fix_leftright_sign(merged_df)

print("Derived markers computed and shoulder distance adjusted.")

Derived markers computed and shoulder distance adjusted.


### Inspect a Species

The interactive 3D plot shows the original cadaver landmarks (blue) and the derived marker positions (red) for a selected species.

In [6]:
# Visualise derived markers for one species
plot_bird_markers(merged_df, row_idx=5)

## Transform Hawk to Target Species

> **§15.2 — Transformation to Different Morphologies**

For each marker, a piecewise linear transformation combines rotation and scaling:

$$T_i = s_i R_i$$

where $s_i$ is the ratio of target-to-source marker norms and $R_i$ is a Rodrigues rotation matrix aligning the two markers. The scaling factor is:

$$s_i = \frac{\|\mathbf{m}_i^{(\text{target})}\|}{\|\mathbf{m}_i^{(\text{hawk})}\|}$$

The rotation angle is:

$$\theta_i = \cos^{-1} \left( \frac{\mathbf{m}_i^{(\text{hawk})} \cdot \mathbf{m}_i^{(\text{target})}}{\|\mathbf{m}_i^{(\text{hawk})}\| \|\mathbf{m}_i^{(\text{target})}\|} \right)$$

These per-marker transformations are assembled into a block-diagonal matrix $T$.

**Tail droop correction (`tail_z_override = -0.05`):** After transformation, the tail-tip z-coordinate is overridden to −0.05 m. Cadaver-derived tail positions tend to droop downwards relative to live birds in flight; this manual correction sets the tail tip to a biologically plausible elevation. The value was chosen by visual inspection of the animated result.

In [7]:
# Select a target species (change index to explore different species)
species_idx = 8
species_name = merged_df['species_common'].iloc[species_idx]
print(f"Target species: {species_name}")

# Transform hawk to target species
approx_bird3d, target_bird3d, transformation_matrix = transform_hawk_to_species(
    hawk3d, species_idx, merged_df
)

# Visualise comparison: hawk (red) vs transformed hawk (blue) vs target (green)
plot_plotly(hawk3d, colour='red')
plot_plotly(approx_bird3d, colour='blue')
plot_plotly(target_bird3d, colour='green')

Target species: mallard


## Transform Morphing Modes and Animate

> **§15.3 — Transformation of Morphing Shape Modes**

The same block-diagonal transformation is applied to the principal components:

$$P^{(\text{transformed})} = T \cdot P$$

Since $T$ is not necessarily orthonormal, the transformed components lose strict orthogonality but preserve the original morphing shape relationships, allowing meaningful comparison across species.

### Pseudoinverse normalisation scheme

To compute species-specific scores we use the Moore–Penrose pseudoinverse of the transformed (and row-normalised) component matrix. The steps are:

1. **Row-normalise** each transformed component vector to unit length, so that the pseudoinverse treats all modes equally regardless of the scaling introduced by $T$.
2. **Compute scores** as $S_{\text{new}} = P_{\text{norm}}^{+} \cdot X^{T}$, where $P_{\text{norm}}^{+}$ is the pseudoinverse of the normalised components.
3. **Rescale** the resulting scores by the original component norms to restore physically meaningful amplitudes.

This approach ensures the morphing shape modes retain correct amplitudes in the new species' coordinate space.

### Animate a single mode on the target species

In [8]:
# Transform principal components using the block-diagonal matrix
pc_transformed = transform_principal_components(principal_components, transformation_matrix)

# --- Pseudoinverse normalisation ---
# Step 1: Compute the norm of each transformed component (used for rescaling)
pc_norms_flat = np.linalg.norm(
    pc_transformed.reshape(pc_transformed.shape[0], -1), axis=1, keepdims=True
)

# Step 2: Row-normalise so the pseudoinverse treats all modes equally
pc_transformed_flat = pc_transformed.reshape(pc_transformed.shape[0], -1)
pc_normalised = pc_transformed_flat / pc_norms_flat

# Step 3: Compute new scores via Moore-Penrose pseudoinverse
# S_new = pinv(P_norm) @ X^T
new_scores = (pinv(pc_normalised) @ scores.T).T

# Step 4: Rescale scores by the original component norms
new_scores = new_scores * pc_norms_flat.T

# Generate animation score range
score_frames = get_score_range(new_scores, num_frames=20)

# Get the transformed mean shape for reconstruction
approx_bird3d.restore_keypoints_to_average()
mu_species = approx_bird3d.right_markers

print(f"Transformed components shape: {pc_transformed.shape}")
print(f"New scores shape: {new_scores.shape}")

Transformed components shape: (12, 12)
New scores shape: (289528, 12)


In [9]:
# Animate Mode 1 (wing lifting) on the target species
colour_list = ['#B5E675', '#6ED8A9', '#51B3D4',
               '#4579AA', '#F19EBA', '#BC96C9',
               '#917AC2', '#BE607F', '#624E8B',
               '#888888', '#888888', '#888888']

component_number = [0]
reconstructed_frames = reconstruct(score_frames, pc_transformed, mu_species, component_number)
recon_bilateral = to_bilateral(reconstructed_frames)

animate_plotly(approx_bird3d, recon_bilateral, colour=colour_list[0])

### Full bilateral flight reconstruction

We can also reconstruct a complete flight sequence using the mean left and right scores from a hawk (Toothless, 9 m straight flight) and project it onto the species morphology.

In [10]:
# Get mean flight scores for a hawk
scores_df, _ = get_score_df(scores, unilateral_frame_info_df, size_bin=0.05)

# Negate PC07 (m-folding) for interpretability: positive = more folded
scores_df["PC07"] = -scores_df["PC07"]

_, Left_mean_scores, _, _ = get_binned_scores(
    scores_df, hawkname="Toothless", perchDist="9m", turn='Straight',
    left=True, year=2020
)
_, Right_mean_scores, _, _ = get_binned_scores(
    scores_df, hawkname="Toothless", perchDist="9m", turn='Straight',
    left=False, year=2020
)

Left_selected = Left_mean_scores.to_numpy()
Right_selected = Right_mean_scores.to_numpy()
components_list = np.arange(12)

# Reconstruct bilateral flight in target species morphology
recon_left = reconstruct(Left_selected, pc_transformed, mu_species, components_list)
recon_right = reconstruct(Right_selected, pc_transformed, mu_species, components_list)
reconstructed_full = to_bilateral(recon_right, left=recon_left)

print(f"Reconstructed flight: {reconstructed_full.shape[0]} frames")
animate_plotly(approx_bird3d, reconstructed_full[:25], colour='#540D6E')

Reconstructed flight: 155 frames


## Summary

The piecewise marker-by-marker transformation successfully projects hawk morphing modes onto diverse avian morphologies:

- **21 species** from the Harvey et al. (2022b) cadaver dataset were transformed, spanning wingspans from 38.5 cm (black swift) to 195 cm (American white pelican).
- The transformed principal components preserve the relative morphing shape patterns whilst adapting to the new morphology.
- Animated flights are **biologically plausible** — the species-specific shapes undergo recognisable wing lifting, spreading, and sweeping motions.
- Highly decorative tail feathers (e.g., pheasants) are not well captured by hawk tail measurements, but these differences could be compared within the same coordinate space.

### Important caveats

- **No flight kinematics data** are available for these species — the animations show each species "flying like a hawk" and do not represent their actual flight behaviour.
- **Cadaver adjustments** (tail width doubling, shoulder correction, tail droop override) are necessary approximations; the transformed shapes should be interpreted qualitatively.
- The pseudoinverse normalisation ensures correct score amplitudes but does not restore orthogonality between transformed components.

This framework provides a **proof of concept** for a shared morphing coordinate space across species. Future work could extend this to species with flight kinematics data, enabling quantitative cross-species comparison of morphing strategies.

---

## References

- Harvey, C., Baliga, V. B., Wong, J. C. M., Altshuler, D. L. & Inman, D. J. (2022). Birds can transition between stable and unstable states via wing morphing. *Nature*, 603, 648–653.
- Harvey, C., De Croon, G., Taylor, G. K. & Bomphrey, R. J. (2023). Lessons from natural flight for aviation: then, now and tomorrow. *Journal of Experimental Biology*, 226, jeb245409.